# Import

In [2]:
using PyCall
using Conda
using Random
using LinearAlgebra
using Distributions
using Statistics
using StatsBase
using Printf
using JuMP
using Gurobi

hmmlearn = pyimport("hmmlearn")
sklearn = pyimport("sklearn")
scipy = pyimport("scipy")
@pyimport hmmlearn.hmm as hmm
@pyimport numpy as np
@pyimport sklearn.tree as tree
@pyimport scipy.stats as stats

┌ Warning: `@pyimport foo` is deprecated in favor of `foo = pyimport("foo")`.
│   caller = _pywrap_pyimport(::PyObject) at PyCall.jl:407
└ @ PyCall /home/victor/.julia/packages/PyCall/ttONZ/src/PyCall.jl:407


# Includes

In [8]:
##############includes################
include("src/set_params_simulator.jl")
include("src/get_samples.jl")
include("src/build_models.jl")
include("src/build_trees.jl")
include("src/build_industrial_faultrees.jl")
include("src/compute_emissionprob.jl")
include("src/set_POMDP_params.jl")
include("src/simulations.jl")
######################################

simulation (generic function with 1 method)

# A tool to create folder if it does not exist yet

In [9]:
##############Tools###################
function createFolder(directory)
    try
        if isdir(directory) == false
            mkdir(directory)
        end
    catch
        println(string("Error:Creating directory ", directory))
    end
end
######################################

createFolder (generic function with 1 method)

# Simulations

In [ ]:
##############Set the parameters######

#Number of components
M = 3

#Maintenance capacity
K = 1

#Maximum number of states
nb_states_max = 10

#Horizon time
Horizon = 200

#Interval time between two maintenance slots
h = 30

#Instance number (if you need to store your results)
instance = 1

#Nb of simulations
#How many seeds to use ?
nb_seeds = 1

#How many simulations given a seed ?
nb_sims = 1
######################################


########Set simulator parameters######
sim_params = set_params_simulator(M)
######################################

##########Build Gaussian HMM############################################################
nb_training_samples = 100
nb_test_samples = 100
sample_size = 8000
nb_states,models = build_models(M,nb_states_max,nb_training_samples,nb_test_samples,sample_size,sim_params)
########################################################################################

##########Build decision trees##########################################################
nb_tests = 1
sample_size = 8000
nb_training_samples = 300
nb_test_samples = 200

dtrees,scores = build_dtrees(nb_training_samples,nb_tests,nb_test_samples,models,sample_size,sim_params)
#TODO : Add a line that restarts our function if the scores are not good enough 


########################################################################################

##########Compute emission probabilities################################################
emissions = build_emissions(dtrees,models)
########################################################################################

#########Build industrial decision trees################################################
nb_tests = 1
sample_size = 8000
nb_training_samples = 300
nb_test_samples = 200
q = 0.90
dtrees_industry = build_industry_dtrees(M,q,nb_training_samples,nb_tests,nb_test_samples,sample_size,sim_params)
#########################################################################################

#################################Set the POMDP parameters#######################################################
costs_f = 10*ones(M)
costs_r = 1*ones(M)

pomdp_params = set_pomdp_parameters(M,h,models,emissions,costs_f,costs_r)
################################################################################################################

##########Start simulations###########

#Cost of policies
cost_industry = 0
fails_industry = 0
repls_industry = 0

cost_MILP_1 = 0
fails_MILP_1 = 0
repls_MILP_1 = 0
time_MILP_1 = 0

cost_MILP_2 = 0
fails_MILP_2 = 0
repls_MILP_2 = 0
time_MILP_2 = 0

cost_MILP_5 = 0
fails_MILP_5 = 0
repls_MILP_5 = 0
time_MILP_5 = 0

cost_LP_1 = 0
fails_LP_1 = 0
repls_LP_1 = 0
time_LP_1 = 0

cost_LP_2 = 0
fails_LP_2 = 0
repls_LP_2 = 0
time_LP_2 = 0

cost_LP_5 = 0
fails_LP_5 = 0
repls_LP_5 = 0
time_LP_5 = 0

for seed in 1:nb_seeds
    Random.seed!(seed)
    for sim in 1:nb_sims
        println("Simulation : ", sim)
        #Initialize the samples for each equipment
        discrete_trajectories = []
        continuous_trajectories = []
        #timeofDeath = zeros(M)
        for m in 1:M
            y,x,Lengths = get_sample_data(rand(sim_params[m].dimension),sample_size,sim_params[m])
            labels = dtrees[m].predict(x)
            # println(labels)
            push!(discrete_trajectories,labels.+1)
            push!(continuous_trajectories,x)
        end
        
        #########################Simulation using policy from industry##############
        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"Industry",dtrees_industry,costs_f,costs_r,
                                                0,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_industry += cost
        global fails_industry += fails
        global repls_industry += repls
        
        #########################Simulation using the linear relaxation of our MILP with valid inequalities####
        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"LP_cuts",dtrees_industry,costs_f,costs_r,
                                                1,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_LP_1 += cost
        global fails_LP_1 += fails
        global repls_LP_1 += repls
        global time_LP_1 += times

        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"LP_cuts",dtrees_industry,costs_f,costs_r,
                                                2,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_LP_2 += cost
        global fails_LP_2 += fails
        global repls_LP_2 += repls
	    global time_LP_2 += times
        
        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"LP_cuts",dtrees_industry,costs_f,costs_r,
                                                5,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_LP_5 += cost
        global fails_LP_5 += fails
        global repls_LP_5 += repls
	    global time_LP_5 += times

        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"MILP",dtrees_industry,costs_f,costs_r,
                                                1,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_MILP_1 += cost
        global fails_MILP_1 += fails
        global repls_MILP_1 += repls
        global time_MILP_1 += times


        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"MILP",dtrees_industry,costs_f,costs_r,
                                                2,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_MILP_2 += cost
        global fails_MILP_2 += fails
        global repls_MILP_2 += repls
        global time_MILP_2 += times

        cost, fails, repls, times = simulation(Horizon,h,sample_size,M,K,sim_params,dtrees,"MILP",dtrees_industry,costs_f,costs_r,
                                                5,pomdp_params,
                                                continuous_trajectories,discrete_trajectories)
        global cost_MILP_5 += cost
        global fails_MILP_5 += fails
        global repls_MILP_5 += repls
        global time_MILP_5 += times      
    end
end

cost_industry = cost_industry/(nb_sims*nb_seeds)
fails_industry = fails_industry/(nb_sims*nb_seeds)
repls_industry = repls_industry/(nb_sims*nb_seeds)

cost_LP_1 = cost_LP_1/(nb_sims*nb_seeds)
fails_LP_1 = fails_LP_1/(nb_sims*nb_seeds)
repls_LP_1 = repls_LP_1/(nb_sims*nb_seeds)
time_LP_1 = time_LP_1/(nb_sims*nb_seeds)

cost_LP_2 = cost_LP_2/(nb_sims*nb_seeds)
fails_LP_2 = fails_LP_2/(nb_sims*nb_seeds)
repls_LP_2 = repls_LP_2/(nb_sims*nb_seeds)
time_LP_2 = time_LP_2/(nb_sims*nb_seeds)

cost_LP_5 = cost_LP_5/(nb_sims*nb_seeds)
fails_LP_5 = fails_LP_5/(nb_sims*nb_seeds)
repls_LP_5 = repls_LP_5/(nb_sims*nb_seeds)
time_LP_5 = time_LP_5/(nb_sims*nb_seeds)

cost_MILP_1 = cost_MILP_1/(nb_sims*nb_seeds)
fails_MILP_1 = fails_MILP_1/(nb_sims*nb_seeds)
repls_MILP_1 = repls_MILP_1/(nb_sims*nb_seeds)
time_MILP_1 = time_MILP_1/(nb_sims*nb_seeds)

cost_MILP_2 = cost_MILP_2/(nb_sims*nb_seeds)
fails_MILP_2 = fails_MILP_2/(nb_sims*nb_seeds)
repls_MILP_2 = repls_MILP_2/(nb_sims*nb_seeds)
time_MILP_2 = time_MILP_2/(nb_sims*nb_seeds)


cost_MILP_5 = cost_MILP_5/(nb_sims*nb_seeds)
fails_MILP_5 = fails_MILP_5/(nb_sims*nb_seeds)
repls_MILP_5 = repls_MILP_5/(nb_sims*nb_seeds)
time_MILP_5 = time_MILP_5/(nb_sims*nb_seeds)

println("Average cost for Heuristic Industry : ", cost_industry)
println("Average number of failures : ", fails_industry)
println("Average number of replacements : ", repls_industry)
println("\n")
println("Average cost for Heuristic LP with T=1 : ", cost_LP_1)
println("Average number of failures : ", fails_LP_1)
println("Average number of replacements : ", repls_LP_1)
println("Average policy time LP with T=1 : ", time_LP_1)
println("\n")
println("Average cost for Heuristic LP with T=2 : ", cost_LP_2)
println("Average number of failures : ", fails_LP_2)
println("Average number of replacements : ", repls_LP_2)
println("Average policy time LP with T=2 : ", time_LP_2)
println("\n")
println("Average cost for Heuristic LP with T=5 : ", cost_LP_5)
println("Average number of failures : ", fails_LP_5)
println("Average number of replacements : ", repls_LP_5)
println("Average policy time LP with T=5 : ", time_LP_5)
println("\n")
println("Average cost for Heuristic MILP with T=1 : ", cost_MILP_1)
println("Average number of failures : ", fails_MILP_1)
println("Average number of replacements : ", repls_MILP_1)
println("Average policy time MILP with T=1 : ", time_MILP_1)
println("\n")
println("Average cost for Heuristic MILP with T=2 : ", cost_MILP_2)
println("Average number of failures : ", fails_MILP_2)
println("Average number of replacements : ", repls_MILP_2)
println("Average policy time MILP with T=2 : ", time_MILP_2)
println("\n")
println("Average cost for Heuristic MILP with T=5 : ", cost_MILP_5)
println("Average number of failures : ", fails_MILP_5)
println("Average number of replacements : ", repls_MILP_5)
println("Average policy time MILP with T=5 : ", time_MILP_5)


#Create the folder of results if it does not exist yet
createFolder(string("results"))
createFolder(string("results/results_",M,"_",K))
outfile = string("results/results_",M,"_",K,"/results_",Horizon,"_",h,"_",instance,".dat")


f = open(outfile, "w")

@printf(f,"Average cost for Heuristic Industry: %.3f \n", cost_industry)
@printf(f,"Average number of failures: %.3f \n", fails_industry)
@printf(f,"Average number of replacements: %.3f\n", repls_industry)
@printf(f,"\n")
@printf(f,"Average cost for Heuristic LP with T=1: %.3f\n", cost_LP_1)
@printf(f,"Average number of failures: %.3f\n", fails_LP_1)
@printf(f,"Average number of replacements: %.3f\n", repls_LP_1)
@printf(f,"Average policy time LP with T=1 : %.3f\n", time_LP_1)
@printf(f,"\n")
@printf(f,"Average cost for Heuristic LP with T=2: %.3f\n", cost_LP_2)
@printf(f,"Average number of failures: %.3f\n", fails_LP_2)
@printf(f,"Average number of replacements: %.3f\n", repls_LP_2)
@printf(f,"Average policy time LP with T=2 : %.3f\n", time_LP_2)
@printf(f,"\n")
@printf(f,"Average cost for Heuristic LP with T=5: %.3f\n", cost_LP_5)
@printf(f,"Average number of failures: %.3f\n", fails_LP_5)
@printf(f,"Average number of replacements: %.3f\n", repls_LP_5)
@printf(f,"Average policy time LP with T=5 : %.3f\n", time_LP_5)
@printf(f,"\n")
@printf(f,"Average cost for Heuristic MILP with T=1: %.3f\n", cost_MILP_1)
@printf(f,"Average number of failures: %.3f\n", fails_MILP_1)
@printf(f,"Average number of replacements: %.3f\n", repls_MILP_1)
@printf(f,"Average policy time MILP with T=1 : %.3f\n", time_MILP_1)
@printf(f,"\n")
@printf(f,"Average cost for Heuristic MILP with T=2: %.3f\n", cost_MILP_2)
@printf(f,"Average number of failures: %.3f\n", fails_MILP_2)
@printf(f,"Average number of replacements: %.3f\n", repls_MILP_2)
@printf(f,"Average policy time MILP with T=2 : %.3f\n", time_MILP_2)
@printf(f,"\n")
@printf(f,"Average cost for Heuristic MILP with T=5: %.3f\n", cost_MILP_5)
@printf(f,"Average number of failures: %.3f\n", fails_MILP_5)
@printf(f,"Average number of replacements: %.3f\n", repls_MILP_5)
@printf(f,"Average policy time MILP with T=5 : %.3f\n", time_MILP_5)


close(f)

Start learning component 1
Train and Tests with 4 states
1
Train and Tests with 5 states
1
2
Train and Tests with 6 states
1
Train and Tests with 7 states
1
Train and Tests with 8 states
1
Train and Tests with 9 states
1
Train and Tests with 10 states
1
Optimal number of states : 7
Start learning component 2
Train and Tests with 4 states
1
Train and Tests with 5 states
1
Train and Tests with 6 states
1
Train and Tests with 7 states
1
Train and Tests with 8 states
1
Train and Tests with 9 states
1
Train and Tests with 10 states
1
Optimal number of states : 9
Start learning component 3
Train and Tests with 4 states
1
2
3